In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
from src.utils.visualization import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 4.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
====================== Hyperparameters =======================
N_EPOCHS: 200
T_MAX: 200
CRITERION: CrossEntropyLoss()
DEVICE: cuda
SEED: 42
BATCH_SIZE: 128
LR: 0.001
MOMENTUM: 0.9
WEIGHT_DECAY: 0.0001
Setting seed to 42


In [2]:
DEBUG = False
SKIP_TRAINING = False
EXP_NAME = "baseline_500"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

Starting experiment baseline_500. DEBUG=False, SKIP_TRAINING=False


In [3]:
resnet = MakeResNet18().to(device)
MODEL_NAME = "ResNet18"

total_params, model_size_mb = get_model_summary(resnet)
print(f"Total Parameters: {total_params:,}")
print(f"Model Size: {model_size_mb:.2f} MB")

Total Parameters: 11,173,962
Model Size: 42.63 MB


In [4]:
trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits(
    n_samples_per_class_train=500)
resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet)

100%|██████████| 170M/170M [00:04<00:00, 35.1MB/s]


Using default 500 samples per class for val.
Original train-val size: 50000
Train size: 5000
Val size: 5000
Test size: 10000
Samples per class (train): Counter({np.int64(4): 500, np.int64(7): 500, np.int64(3): 500, np.int64(1): 500, np.int64(2): 500, np.int64(8): 500, np.int64(5): 500, np.int64(0): 500, np.int64(9): 500, np.int64(6): 500})
Samples per class (val): Counter({np.int64(5): 500, np.int64(8): 500, np.int64(2): 500, np.int64(9): 500, np.int64(1): 500, np.int64(3): 500, np.int64(0): 500, np.int64(6): 500, np.int64(7): 500, np.int64(4): 500})
T_MAX: 200


In [5]:
if not SKIP_TRAINING:
    train_model(
        model=resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        device=device,
        experiment_name=EXP_NAME,
        model_name=MODEL_NAME,
        val_accuracy_storing_threshold=40,
        DEBUG=DEBUG
    )

Model weights will be saved to: /kaggle/working/artifacts/checkpoints/baseline_500_ResNet18.pth
Stats will be saved to: /kaggle/working/artifacts/stats/baseline_500_ResNet18.pkl

=== Starting Training: ResNet18 with 200 epochs ===
Epoch 1/200 | Loss: 3.018 | Val Acc: 16.70%
Epoch 2/200 | Loss: 2.210 | Val Acc: 19.62%
Epoch 3/200 | Loss: 2.065 | Val Acc: 27.72%
Epoch 4/200 | Loss: 1.920 | Val Acc: 28.46%
Epoch 5/200 | Loss: 1.887 | Val Acc: 28.52%
Epoch 6/200 | Loss: 1.793 | Val Acc: 29.96%
Epoch 7/200 | Loss: 1.768 | Val Acc: 36.04%
Epoch 8/200 | Loss: 1.684 | Val Acc: 37.28%
Epoch 9/200 | Loss: 1.680 | Val Acc: 37.10%
Epoch 10/200 | Loss: 1.684 | Val Acc: 36.12%
    --> New Best Saved: 41.50%
Epoch 11/200 | Loss: 1.621 | Val Acc: 41.50%
Epoch 12/200 | Loss: 1.593 | Val Acc: 40.70%
Epoch 13/200 | Loss: 1.611 | Val Acc: 33.74%
    --> New Best Saved: 42.70%
Epoch 14/200 | Loss: 1.585 | Val Acc: 42.70%
Epoch 15/200 | Loss: 1.514 | Val Acc: 40.78%
    --> New Best Saved: 45.08%
Epoch 16/2

In [6]:
debug_suff = "_DEBUG" if DEBUG else ""
load_weights(resnet, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
if not SKIP_TRAINING:
    print(f'Final test accuracy is: {calculate_accuracy(resnet, testloader, device):.3f}')

Final test accuracy is: 74.980
